In [1]:
import json
import os
import pandas
import json

def GetPositions(dir):
    return [(item, os.path.join(dir, item).replace("\\", "/")) for item in os.listdir(dir)]

def CheckPositionResponse(item, role, round):
    if os.path.exists(f"Results/response_{item}_{role}_{round}.json"):
        return True

def LoadPositionResponse(item, role, round):
    with open(f"Results/response_{item}_{role}_{round}.json", 'r') as f:
        return json.load(f)

def LoadPosition(position):
    position_item, position_path = position[0], position[1]
    results = []
    for role in ["female", "male", "local", "non-local", "experience", "non-experience"]:
        for i in range(1, 3):
            if CheckPositionResponse(position_item, role, i):
                results.append((int(position_item), role, i, LoadPositionResponse(position_item, role, i)))
    return results

def FormatResult(result):
    response = json.loads(result[3]["output"]["choices"][0]["message"]["content"][0]["text"].replace("```json", "").replace("```", ""))
    return {
        "item": result[0], 
        "role": result[1], 
        "round": result[2], 
        "score": response["score"], 
        "reason": response["reason"]
    }

In [2]:
data_root = "./Data/StreetViews"
positions = GetPositions(data_root)
results = []
for position in positions:
    results.extend(LoadPosition(position))
results = pandas.DataFrame([FormatResult(result) for result in results]); results

,item,role,round,score,reason
0,1,female,1,65,1. **人身安全受到威胁的可能性**：该街景显示了一个城市环境，有建筑物、停车场和道路。虽...
1,1,female,2,75,1. **人身安全受到威胁的可能性**：从图片来看，该区域的街道较为宽敞，车辆行驶有序，没有...
2,1,male,1,75,1. **人身安全受到威胁的可能性**：从图片来看，该地区街道较为宽敞，车辆行驶有序，没有明...
3,1,male,2,75,1. **人身安全受到威胁的可能性**：从四张街景图来看，该区域的交通状况较为有序，车辆停放...
4,1,local,1,75,1. **人身安全受到威胁的可能性**：该街景显示的区域看起来是一个城市中的普通商业和住宅区...
...,...,...,...,...,...
12043,999,non-local,2,75,1. **人身安全受到威胁的可能性**：从图片来看，该街景位于一个城市中的普通商业区，街道较...
12044,999,experience,1,65,1. **人身安全受到威胁的可能性**：该街景显示的是一个典型的市区街道环境，存在一定的车辆...
12045,999,experience,2,65,1. **人身安全受到威胁的可能性**：该街景显示的是一个典型的市区街道环境，存在一定的车辆...
12046,999,non-experience,1,75,根据提供的四个方位的街景图，可以综合分析该处街景的安全感如下：\n\n1. **人身安全受到...


In [3]:
score_results = results.drop(columns = ["reason"]).pivot_table(values = "score", index = ["item", "role"], columns = "round").reset_index()
score_results["diff"] = abs(score_results[1] - score_results[2])
score_results["score"] = (score_results[1] + score_results[2]) / 2
score_results["role"] = score_results["role"].apply(lambda x: x.replace("experience", "issue"))
score_results.to_csv("vlm_reliability.csv", index=False); score_results

round,item,role,1,2,diff,score
0,1,issue,65,65,0,65.0
1,1,female,65,75,10,70.0
2,1,local,75,75,0,75.0
3,1,male,75,75,0,75.0
4,1,non-issue,75,75,0,75.0
...,...,...,...,...,...,...
6019,1005,female,65,65,0,65.0
6020,1005,local,65,65,0,65.0
6021,1005,male,65,65,0,65.0
6022,1005,non-issue,75,75,0,75.0


In [4]:
pandas.crosstab(score_results["diff"], score_results["role"])

role,female,issue,local,male,non-issue,non-local
diff,,,,,,
0,982,981,994,989,987,989
5,0,0,0,0,1,0
10,22,23,10,15,16,15


In [5]:
score_results = score_results.query("diff <= 5")

In [6]:
score_results.describe()

round,item,1,2,diff,score
count,5923.000000,5923.000000,5923.000000,5923.000000,5923.000000
mean,502.948337,71.103326,71.104170,0.000844,71.103748
std,289.671397,8.025886,8.027611,0.064968,8.026683
min,1.000000,35.000000,35.000000,0.000000,35.000000
25%,252.500000,65.000000,65.000000,0.000000,65.000000
50%,502.000000,75.000000,75.000000,0.000000,75.000000
75%,753.000000,75.000000,75.000000,0.000000,75.000000
max,1005.000000,85.000000,90.000000,5.000000,87.500000


In [7]:
score_results[["item", "role", "score"]].to_csv("vlm_score.csv", index = False)